In [21]:
import pandas as pd
import statistics as st
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

df = pd.read_csv("../data/raw/train.csv")

In [2]:
df["IsFemale"] = df["Sex"] == "female"
df["FamilySize"] = df["SibSp"] + df["Parch"] + 1

# Build Title and finish ALL text-based work on it first
df["Title"] = df["Name"].str.extract(r",\s*([^\.]+)\.")
df.loc[~df["Title"].isin(["Mr", "Miss", "Mrs", "Master"]), "Title"] = "Rare"

# Fill Age while Title is still plain text (needed for groupby)
df["Age"] = df["Age"].fillna(df.groupby("Title")["Age"].transform("median"))

# Fill Embarked before encoding it
df["Embarked"] = df["Embarked"].fillna("S")

# NOW one-hot encode both, once everything that needed the plain versions is done
df = pd.get_dummies(df, columns=["Title"], prefix="Title")
df = pd.get_dummies(df, columns=["Embarked"], prefix="Embarked")

# HasCabin doesn't depend on ordering relative to the above
df["HasCabin"] = df["Cabin"].notna()

In [3]:
feature_cols = [
    "IsFemale", "Pclass", "Age", "Fare", "FamilySize", "HasCabin",
    "Title_Master", "Title_Miss", "Title_Mr", "Title_Mrs", "Title_Rare",
    "Embarked_C", "Embarked_Q", "Embarked_S"
]
X = df[feature_cols]
y = df["Survived"]

# 11. Split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

In [4]:
X.isna().sum()

IsFemale        0
Pclass          0
Age             0
Fare            0
FamilySize      0
HasCabin        0
Title_Master    0
Title_Miss      0
Title_Mr        0
Title_Mrs       0
Title_Rare      0
Embarked_C      0
Embarked_Q      0
Embarked_S      0
dtype: int64

In [5]:
X.shape

(891, 14)

In [6]:
X.head()

,IsFemale,Pclass,Age,Fare,FamilySize,HasCabin,Title_Master,Title_Miss,Title_Mr,Title_Mrs,Title_Rare,Embarked_C,Embarked_Q,Embarked_S
0,False,3,22.0,7.2500,2,False,False,False,True,False,False,False,False,True
1,True,1,38.0,71.2833,2,True,False,False,False,True,False,True,False,False
2,True,3,26.0,7.9250,1,False,False,True,False,False,False,False,False,True
3,True,1,35.0,53.1000,2,True,False,False,False,True,False,False,False,True
4,False,3,35.0,8.0500,1,False,False,False,True,False,False,False,False,True


In [7]:
from sklearn.tree import DecisionTreeClassifier

tree_model0 = DecisionTreeClassifier(max_depth=4, random_state=42)
tree_model0.fit(X_train, y_train)

tree_predictions = tree_model0.predict(X_val)
accuracy_score(y_val, tree_predictions)

0.8156424581005587

In [8]:
from sklearn.tree import DecisionTreeClassifier

random_states = [1, 7, 42, 55, 99]
depths_to_try = [2, 3, 4, 5, 6, 8]
results = {}

for depth in depths_to_try:
    scores = []
    for rs in random_states:
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=rs
        )
        tree_model = DecisionTreeClassifier(max_depth=depth, random_state=rs)
        tree_model.fit(X_train, y_train)
        acc = accuracy_score(y_val, tree_model.predict(X_val))
        scores.append(acc)
    results[depth] = scores
    print(f"depth={depth}: mean={sum(scores)/len(scores)*100:.2f}%, min={min(scores)*100:.2f}%, max={max(scores)*100:.2f}%")

depth=2: mean=79.22%, min=76.54%, max=83.24%
depth=3: mean=81.90%, min=78.21%, max=84.36%
depth=4: mean=81.79%, min=80.45%, max=83.24%
depth=5: mean=80.00%, min=78.21%, max=81.56%
depth=6: mean=80.45%, min=79.33%, max=81.56%
depth=8: mean=80.00%, min=78.77%, max=81.56%


In [9]:
importances = pd.Series(tree_model.feature_importances_, index=X.columns)
importances.sort_values(ascending=False)

Title_Mr        0.409223
Fare            0.213974
Age             0.158503
Pclass          0.080531
FamilySize      0.045473
Title_Rare      0.040816
IsFemale        0.020982
HasCabin        0.016974
Title_Miss      0.005079
Title_Mrs       0.004575
Embarked_C      0.003870
Title_Master    0.000000
Embarked_Q      0.000000
Embarked_S      0.000000
dtype: float64

In [11]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
tree_model = DecisionTreeClassifier(max_depth=3, random_state=42)
tree_model.fit(X_train, y_train)

importances = pd.Series(tree_model.feature_importances_, index=X.columns)
importances.sort_values(ascending=False)

Title_Mr        0.637112
Pclass          0.157287
FamilySize      0.083419
HasCabin        0.050947
Title_Rare      0.050510
Age             0.013116
Fare            0.007608
IsFemale        0.000000
Title_Miss      0.000000
Title_Master    0.000000
Title_Mrs       0.000000
Embarked_C      0.000000
Embarked_Q      0.000000
Embarked_S      0.000000
dtype: float64

In [18]:
from sklearn.ensemble import RandomForestClassifier

forest_model = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=42)
forest_model.fit(X_train, y_train)

forest_predictions = forest_model.predict(X_val)
accuracy_score(y_val, forest_predictions)

0.7988826815642458

In [19]:
random_states = [1, 7, 42, 55, 99]
scores_forest = []

for rs in random_states:
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=rs)
    forest_model = RandomForestClassifier(n_estimators=100, max_depth=3, random_state=rs)
    forest_model.fit(X_train, y_train)
    acc = accuracy_score(y_val, forest_model.predict(X_val))
    scores_forest.append(acc)

print(scores_forest)

[0.8324022346368715, 0.8044692737430168, 0.7988826815642458, 0.8435754189944135, 0.8547486033519553]


In [20]:
importances = pd.Series(forest_model.feature_importances_, index=X.columns)
importances.sort_values(ascending=False)

Title_Mr        0.304396
IsFemale        0.220352
Fare            0.091454
Pclass          0.082332
Title_Mrs       0.081884
Title_Miss      0.055284
HasCabin        0.050019
FamilySize      0.047257
Age             0.035611
Title_Master    0.009425
Embarked_S      0.009016
Embarked_C      0.009001
Title_Rare      0.002019
Embarked_Q      0.001951
dtype: float64

In [26]:
max(scores_forest)

0.8547486033519553

In [27]:
random_states = [1, 7, 42, 55, 99]
depths_to_try = [3, 5, 7, None]   # None = no depth limit at all
n_trees_to_try = [50, 100, 200]

for depth in depths_to_try:
    for n_trees in n_trees_to_try:
        scores = []
        for rs in random_states:
            X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=rs)
            m = RandomForestClassifier(n_estimators=n_trees, max_depth=depth, random_state=rs)
            m.fit(X_train, y_train)
            scores.append(accuracy_score(y_val, m.predict(X_val)))
        print(f"depth={depth}, n_trees={n_trees}: mean={st.mean(scores)*100:.2f}%, stdev={st.stdev(scores)*100:.2f}")

depth=3, n_trees=50: mean=82.79%, stdev=2.25
depth=3, n_trees=100: mean=82.68%, stdev=2.44
depth=3, n_trees=200: mean=82.35%, stdev=2.19
depth=5, n_trees=50: mean=83.02%, stdev=1.75
depth=5, n_trees=100: mean=83.80%, stdev=1.89
depth=5, n_trees=200: mean=84.25%, stdev=2.45
depth=7, n_trees=50: mean=82.57%, stdev=2.54
depth=7, n_trees=100: mean=82.79%, stdev=1.99
depth=7, n_trees=200: mean=83.13%, stdev=1.45
depth=None, n_trees=50: mean=79.66%, stdev=1.16
depth=None, n_trees=100: mean=80.11%, stdev=1.29
depth=None, n_trees=200: mean=80.00%, stdev=0.73
